# Capstone companion --- Chapter 4: Tasks, State and Actions

Chapter~4 introduces the three typed objects that make an agent auditable. A `TaskSpec` states what is to be solved, together with its inputs, expected outputs, constraints and validation rules. An `AgentState` records how far the agent has progressed --- the step count, the messages, the tool results, the status. An `Action` is the single thing the agent emits on each step, drawn from a small closed union. Because each object is a validated schema, a trajectory can be serialized, replayed and inspected after the fact. This companion reads those objects on the capstone banking complaint agent.

The capstone agent handles a customer complaint through a fixed workflow: classify, extract facts, search policy, flag regulatory risk, then draft or escalate. Each of the three Chapter~4 objects has a concrete role in that workflow. The `TaskSpec` names the complaint to be handled; the `AgentState` accumulates the result of each tool as the workflow advances; and the `Action` the agent proposes at a given step is a `ToolCall`, an `Escalate` or a `Finish`, according to what the state so far permits.

In [ ]:
from agentlab.core import (
    TaskSpec, ValidationRule,
    AgentState,
    Action, ActionKind, ToolCall, AskUser, Finish, Escalate,
    parse_action,
)

## The TaskSpec for a complaint

A task is more than a prompt string. The `TaskSpec` names the goal, carries the customer message as a typed input, and declares what a valid result must contain and what it must not do. The constraints and validation rules are the contract the finished trajectory is checked against; they are recorded on the task itself rather than left implicit in the agent's code.

In [ ]:
message = (
    'I was charged a $35 overdraft fee on my checking account that I never '
    'authorized, and the bank refuses to reverse it.'
)
task = TaskSpec(
    goal='handle complaint',
    inputs={'message': message},
    expected_outputs=['classification', 'issue', 'risk_flags', 'draft_response'],
    constraints=[
        'cite a policy for any factual claim in the reply',
        'do not promise a fee reversal without authorization',
    ],
    validation=[
        ValidationRule(name='grounded', description='every claim carries an evidence pointer'),
        ValidationRule(name='no_pii_leak', description='reply contains no account or card numbers'),
    ],
)
print('goal        :', task.goal)
print('inputs      :', list(task.inputs))
print('expected    :', task.expected_outputs)
print('constraints :', task.constraints)
print('validation  :', [r.name for r in task.validation])

The goal field is validated on construction: it may not be empty. This is the smallest example of the chapter's discipline --- a malformed task is rejected at the boundary rather than carried silently into the loop.

In [ ]:
from pydantic import ValidationError

try:
    TaskSpec(goal='   ', inputs={'message': message})
except ValidationError as e:
    print('rejected empty goal:', e.errors()[0]['msg'])

## The AgentState the agent evolves

The state records where the agent is, not where a conversation is. It is initialized from the task with an empty history, and each executed step appends a tool result and advances the step counter. Because every field is a plain serializable value, the whole state can be written to disk and reloaded without loss --- the property that makes replay and audit possible.

In [ ]:
state = AgentState(task=task)
print('step        :', state.step)
print('status      :', state.status)
print('tool_results:', state.tool_results)

# The workflow advances by appending tool results and stepping. Here we simulate
# the first two nodes (classify, extract) to show the shape the agent reads.
state.tool_results.append({'success': True, 'output': {'category': 'billing_dispute', 'confidence': 0.91}})
state.step = 1
state.tool_results.append({'success': True, 'output': {'issue': 'unauthorized_fee', 'product': 'checking'}})
state.step = 2
print('after two steps -> step', state.step, 'results', len(state.tool_results))

The state round-trips through a dictionary without loss. `to_dict` and `from_dict` are the serialization boundary: an audit log stores the dictionary, and a later process reconstructs the exact `AgentState` to inspect or resume.

In [ ]:
restored = AgentState.from_dict(state.to_dict())
print('round-trip equal    :', restored == state)
print('restored goal       :', restored.task.goal)
print('restored step       :', restored.step)
print('restored result[0]  :', restored.tool_results[0]['output'])

## The Action union the agent emits

On each step the agent emits exactly one `Action`. The union is closed: a `ToolCall` invokes a named tool with arguments, an `AskUser` requests missing information, a `Finish` returns the compiled result, and an `Escalate` hands the case to a human with a reason. Each variant is discriminated by a `kind` field constrained to a single literal, so an action deserialized from an audit log parses back to the correct type.

In [ ]:
actions = [
    ToolCall(tool_name='classify_complaint', arguments={'message': message}),
    AskUser(question='Which account was the fee charged to?'),
    Escalate(reason='regulatory risk flagged: UDAAP', context={'flags': ['UDAAP']}),
    Finish(output={'recommended_action': 'respond'}),
]
for a in actions:
    print(f'{a.kind:12s} -> {type(a).__name__}')
print()
print('ActionKind members:', [k.value for k in ActionKind])

The `kind` literal is what makes the union safe across the serialization boundary. `parse_action` dispatches on that field to reconstruct the concrete type from a plain dictionary, and the actions are frozen, so an action recorded in the trajectory cannot be mutated after it is emitted.

In [ ]:
logged = ToolCall(tool_name='search_policy', arguments={'query': message}).model_dump()
print('logged dict :', logged)
reparsed = parse_action(logged)
print('reparsed    :', type(reparsed).__name__, '- tool', reparsed.tool_name)

try:
    reparsed.tool_name = 'other'      # actions are frozen
except ValidationError as e:
    print('frozen      :', e.errors()[0]['type'])

## The three objects in the capstone loop

The complaint agent's decision rule is a pure function of the state: it reads the step count and the tool results so far, and returns the next `Action`. The cell below runs that rule directly on the two-step state assembled above, without executing any tool, to show which action the agent proposes given how far the workflow has advanced.

In [ ]:
from agentlab.capstone.complaint_agent import ComplaintAgent

agent = ComplaintAgent()

s0 = AgentState(task=task)
print('step 0 ->', repr(agent.propose_action(s0)))

s1 = AgentState(task=task, step=1,
                tool_results=[{'success': True, 'output': {'category': 'billing_dispute', 'confidence': 0.91}}])
print('step 1 ->', repr(agent.propose_action(s1)))

# A recorded tool failure short-circuits the same rule to an Escalate.
s_fail = AgentState(task=task, step=1,
                    tool_results=[{'success': False, 'error': 'PII detected in message'}])
print('failure->', repr(agent.propose_action(s_fail)))

This is the capstone's realization of Chapter~4. The `TaskSpec` fixes the problem and the contract it will be judged against; the `AgentState` carries the trajectory in a form that serializes, replays and audits without loss; and the `Action` union names the closed set of things the agent may do at each step, each variant typed and discriminated so it survives the round-trip through the log. Chapter~5 gives the tools that a `ToolCall` invokes their own typed input and output schemas, and the capstone chapter (Chapter~15) assembles the full governed workflow over these objects.